# K-Nearest Neighbors Lesson (5/28/26)
---------------------------------------------------------------------------------------------------------------------------------------------------------

**K-Nearest Neighbors Classifier**

K-Nearest Neighbors Classifier

  - KNN is a classification algorithm — its core idea is that data points with similar attributes tend to fall into
  similar categories.
  - Data points are plotted by their attributes — e.g., each point has an x value and a y value, letting it be placed on
  a graph.
  - Color/label represents the class the algorithm is trying to predict (e.g., green or red).
  - White (uncolored) points have no class yet — classifying these unknown points is the algorithm's purpose.
  - k = the number of nearest neighbors the algorithm looks at to make a classification.
  - The value of k can change the result:
    - k = 3: smaller circle → 2 green, 1 red → classified as green
    - k = 5: larger circle → 3 red, 2 green → classified as red
  - Core process: given a dataset of points with known classes, take a new point with an unknown class, find its nearest
  neighbors, and classify it based on them.


**Introduction**


  - Feature — a piece of information associated with a data point.
  - Example dataset: movies. Potential numeric features of a movie:
    - Length of the movie (in minutes)
    - Budget of the movie (in dollars)
  - Numeric features let you place movies in a multi-dimensional space (like the 2D graph from before).
  - Boolean features — features that are either True or False. Examples:
    - Black and white → True for B&W movies, False otherwise
    - Directed by Stanley Kubrick → True only for his films, False for almost all others
  - Classification goal: Label each movie as good or bad.
    - "Good" = IMDb rating of 7.0 or greater → class 1
    - "Bad" = below 7.0 → class 0
  - Example data point format: [length, budget, directed_by_kubrick]

In [1]:
#Ex.) 

#Movie arrays 
mean_girls = [97, 17000000, False]
the_shining = [146, 19000000, True]
gone_with_the_wind = [238, 3977000, False]

**Distance Between Points - 2D**


  - Why we need a distance formula: humans can eyeball nearest neighbors on a graph, but a computer needs a precise
  definition of "close" vs. "far apart."
  - Solution: use the Distance Formula to measure how far apart two points are.
  - Example uses 2 dimensions:
    - Length of the movie
    - Movie's release date
  - Example data points:
    - Star Wars → 125 minutes, released 1977
    - Raiders of the Lost Ark → 115 minutes, released 1981
  - 2D distance formula:

  $$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$


In [2]:
#Ex.) 

#Distance Formula Function
def distance(movie1, movie2):
  dist = ((movie1[0] - movie2[0])**2 + (movie1[1] - movie2[1])**2)**0.5
  return dist

#List of movies with arguments (Runtime, Release Date)
star_wars = [125, 1977]
raiders = [115, 1981]
mean_girls = [97, 2004]

#Calling and executing a function 
print("Distance of SW-Raiders: ",  distance(star_wars, raiders))
print("Distance of SW-MG: ",  distance(star_wars, mean_girls))

Distance of SW-Raiders:  10.770329614269007
Distance of SW-MG:  38.897300677553446


**Distance Between Points - 3D & ND**

 - Limitation of 2D: using only length and release date is restrictive — there's much more useful movie data available.
  - Add a third dimension: e.g., the movie's budget, requiring distance to be measured in 3D space.
  - 3D distance formula:

  $$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + (z_1 - z_2)^2}$$

  - Beyond 3D: points become impossible to visualize past 3 dimensions, but distance can still be calculated.
  - Generalized distance formula between points $A$ and $B$ in $N$-dimensional space:

  $$d = \sqrt{(A_1 - B_1)^2 + (A_2 - B_2)^2 + \cdots + (A_n - B_n)^2}$$

  - $A_1 - B_1$ = difference between the first feature of each point
  - $A_n - B_n$ = difference between the last feature of each point
  - Takeaway: this lets us find the K-Nearest Neighbors of a point in any number of dimensions, using as many movie
  features as we want.
  - These distances will later be used to find the nearest neighbors of an unlabeled point.

In [6]:
#Ex.) 

star_wars = [125, 1977, 11000000]
raiders = [115, 1981, 18000000]
mean_girls = [97, 2004, 17000000]

def distance1(movie1, movie2):
  squared_difference = 0
  for i in range(len(movie1)):
    squared_difference += (movie1[i] - movie2[i])**2
  Final_Dist = (squared_difference)**0.5
  return Final_Dist 

print("Distance of SW-Raiders: ",  distance1(star_wars, raiders))
print("Distance of SW-MG: ",  distance1(star_wars, mean_girls))

Distance of SW-Raiders:  7000000.000008286
Distance of SW-MG:  6000000.000126083


**Data with Different Scales: Normalization**


- The three steps of the KNN algorithm:
    a. Normalize the data
    b. Find the k nearest neighbors
    c. Classify the new point based on those neighbors
  - The problem — dimensions have very different scales:
    - Release dates differ by at most ~125 years
    - Budgets can differ by millions of dollars
  - Why this breaks KNN: the distance formula treats all dimensions equally regardless of scale.
    - A 1-year difference is treated as equal to a $1 budget difference — absurd.
    - The large-scale feature (budget) dominates and drowns out all other dimensions, making them essentially
  meaningless.
  - The solution — normalize the data so every value falls between 0 and 1.
  - This lesson uses min-max normalization.
  - Min-max normalization formula:

  $$x_{norm} = \frac{x - \min}{\max - \min}$$

In [7]:
#Ex.) 

release_dates = [1897.0, 1998.0, 2000.0, 1948.0, 1962.0, 1950.0, 1975.0, 1960.0, 2017.0, 1937.0, 1968.0, 1996.0, 1944.0, 1891.0, 1995.0, 1948.0, 2011.0, 1965.0, 1891.0, 1978.0]

def min_max_normalize(lst):
    minimum = min(lst)
    maximum = max(lst)
    normalized = []
    for value in lst:
      normalized.append((value - minimum) / (maximum - minimum))
    return normalized

print(min_max_normalize(release_dates))

[0.047619047619047616, 0.8492063492063492, 0.8650793650793651, 0.4523809523809524, 0.5634920634920635, 0.46825396825396826, 0.6666666666666666, 0.5476190476190477, 1.0, 0.36507936507936506, 0.6111111111111112, 0.8333333333333334, 0.42063492063492064, 0.0, 0.8253968253968254, 0.4523809523809524, 0.9523809523809523, 0.5873015873015873, 0.0, 0.6904761904761905]


  - Because 1891 is the minimum and 2017 is the maximum in the list. Min-max
    normalization maps the smallest value to 0 and the largest to 1. Since 1897 is only 6 years above the minimum (out of
    a 126-year total range), it lands very close to 0.